# 2026.10.14(수) 19h~

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
InputNo = 10
# 데이터 생성 함수
def generate_data(num_samples):
    X = np.random.randint(0, 10, (num_samples, InputNo))
    y = np.flip(X, axis=1)
    return X, y

# 데이터 생성
num_samples = 20000
X, y = generate_data(num_samples)

# 데이터 형태 변환 (LSTM을 사용하기 위해 3D 텐서로 변환)
X = X.reshape((X.shape[0], X.shape[1], 1))
y = y.reshape((y.shape[0], y.shape[1], 1))

2026-10-14 19:14:17.954443: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


> **질문 (4교시)**: LSTM 기반으로 sequence를 만들 때 `return_state`, `return_sequences` 각각의 개념, 역할/기능, 차이점 등을 설명해 보기

## `return_sequences`와 `return_state` 정리

### 먼저: LSTM이 내부에 가진 것
LSTM은 시퀀스를 앞에서부터 한 칸(time step)씩 읽으면서 두 가지를 갱신한다.
- **h (hidden state, 은닉 상태)**: 그 시점까지 읽은 내용을 요약한 "현재 출력용 기억". 매 시점의 출력이 곧 h다.
- **c (cell state, 셀 상태)**: LSTM 내부에 따로 가진 "장기 기억". 바깥으로는 보통 드러나지 않는다.

한 시점을 읽을 때마다 (이전 h, 이전 c, 현재 입력) → (새 h, 새 c)로 바뀐다.

### 1. `return_sequences` — "출력을 몇 개 줄까?"
| 값 | 반환하는 것 | 출력 shape |
|---|---|---|
| `False` (기본값) | **마지막 시점의 출력 하나**만 | `(batch, units)` |
| `True` | **모든 시점의 출력** | `(batch, time_steps, units)` |

- `False`는 시퀀스 전체를 읽고 "하나의 결론"만 필요할 때 쓴다. 예: 문장 → 긍정/부정 분류.
- `True`는 **시점마다 결과가 필요**할 때 쓴다. 예: 숫자 10개를 읽고 **위치마다** 숫자를 하나씩 내놓는 역순 출력, LSTM 층을 여러 개 쌓을 때(다음 층이 시퀀스를 입력으로 받아야 함).

### 2. `return_state` — "마지막 기억(h, c)도 따로 줄까?"
| 값 | 반환하는 것 |
|---|---|
| `False` (기본값) | 출력만 (상태는 안 줌) |
| `True` | 출력 **+ 마지막 시점의 h + 마지막 시점의 c** (총 3개) |

- 이 상태를 받아 두면 **다른 LSTM의 시작 기억으로 넘겨줄 수 있다** (`initial_state=[h, c]`).
- `return_state=True`는 반환 값이 늘어나므로 받는 변수도 3개가 되어야 한다.

### 3. 둘의 차이와 조합
- `return_sequences` : **출력(시간 축)** 을 전부 줄지 마지막 하나만 줄지 결정
- `return_state` : **마지막 내부 기억(h, c)** 을 따로 꺼내 줄지 결정
- 서로 독립이라 4가지 조합이 가능하다.

| `return_sequences` | `return_state` | 돌려주는 값 |
|---|---|---|
| False | False | 마지막 출력 1개 |
| True | False | 모든 시점의 출력 |
| False | True | 마지막 출력, h, c  (여기서 마지막 출력 = h) |
| True | True | 모든 시점의 출력, h, c  (여기서 마지막 시점의 출력 = h) |

> 참고: 출력은 결국 h이므로 "마지막 출력"과 "h"는 같은 값이다. `return_state=True`로 새로 얻는 정보는 사실상 **c**다.

### 4. 이 노트북 코드에서 어떻게 쓰였나
```python
# 인코더: 숫자 10개를 읽고 "요약(h, c)"만 필요하다
encoder = LSTM(128, return_state=True)                  # return_sequences는 기본값 False
encoder_outputs, state_h, state_c = encoder(encoder_inputs)
encoder_states = [state_h, state_c]                     # 입력 전체를 압축한 기억

# 디코더: 그 기억에서 출발해, 위치마다 숫자를 하나씩 내놓는다
decoder_lstm = LSTM(128, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = Dense(10, activation='softmax')         # 위치마다 0~9 중 하나를 고른다
```
- **인코더**: `return_state=True` → 마지막 h, c를 꺼낸다. 여기서 출력(`encoder_outputs`)은 쓰지 않고, **h와 c가 "입력 시퀀스를 요약한 정보"** 역할을 한다. 시점별 출력은 필요 없어서 `return_sequences`는 기본값(False)이다.
- **디코더**: `initial_state=encoder_states` → 인코더가 만든 기억을 **시작 상태**로 받는다. `return_sequences=True` → 10개 위치 **모두**의 출력이 필요하다(위치마다 Dense로 숫자를 고르므로). 반환된 상태 두 개는 `_`로 버렸다. 이 학습 코드에서는 필요 없기 때문이다. (한 글자씩 생성하는 추론 방식에서는 이 상태를 다음 단계에 넘겨줘야 하므로 필요해진다.)
- **즉 이어지는 흐름**: `return_state`로 인코더의 최종 기억을 꺼냄 → `initial_state`로 디코더에 전달 → `return_sequences`로 디코더가 위치별 출력을 냄.

### 5. 앞 노트북(1_1_10, 3번)과 비교
- 앞에서는 `Bidirectional(LSTM(64, return_sequences=True))` 한 층만 썼다. 입력과 출력 길이가 같아서 **위치별 출력(`return_sequences=True`)만 있으면** 됐고, 상태를 다른 층에 넘길 필요가 없었다.
- 이번 Seq2Seq(인코더-디코더)는 인코더를 따로 두고 **상태(h, c)를 디코더에 넘겨주는 구조**라서 `return_state`가 필요하다. 이 구조는 입력과 출력 길이가 달라도(번역, 요약 등) 쓸 수 있다는 점이 장점이다.

### 한 줄 요약 (발표/답변용)
> "`return_sequences`는 LSTM이 **모든 시점의 출력**을 내보낼지 마지막 하나만 내보낼지를 정하고, `return_state`는 **마지막 시점의 h·c 상태**를 따로 반환할지를 정합니다. Seq2Seq에서는 인코더가 `return_state=True`로 입력을 요약한 상태를 꺼내 디코더의 `initial_state`로 넘기고, 디코더는 `return_sequences=True`로 시점마다 출력을 만들어 냅니다."

### 이 코드에서 눈에 띄는 점 (참고)
- 디코더 입력으로 `np.zeros_like(X)`(전부 0)를 넣는다. 그래서 디코더는 정답 힌트 없이 **인코더가 넘겨준 h, c 만으로** 역순 숫자를 만들어 내야 한다.
- `InputNo = 10`(숫자 10개)인데 아래쪽 주석 처리된 입력 루프는 5개를 기준으로 쓰여 있어서 값이 맞지 않는다. 지금은 주석(`'''`) 안에 있어 실행되지 않는다.

---


## `sparse_categorical_crossentropy` vs `categorical_crossentropy`

강사님 질문의 이름 그대로 맞다. 두 손실 함수는 **다중 분류**(여러 클래스 중 하나 고르기)에서 쓰고, **계산하는 값은 똑같고 "정답을 어떤 형태로 주느냐"만 다르다.**

### 한 줄 차이
| | `categorical_crossentropy` | `sparse_categorical_crossentropy` |
|---|---|---|
| 정답(y) 형태 | **one-hot 벡터** | **정수 번호** (클래스 번호) |
| 정답 예시 (클래스 3, 클래스 수 10) | `[0,0,0,1,0,0,0,0,0,0]` | `3` |
| 이 노트북의 y shape | `(20000, 10, 10)` | `(20000, 10, 1)` ← **지금 사용** |
| 만드는 방법 | `to_categorical(y, num_classes=10)` | 별도 변환 없이 숫자 그대로 |

### 공통점
- 모델 출력은 둘 다 같다: 마지막 층 `Dense(10, activation='softmax')`가 클래스 10개의 확률을 낸다.
- 계산도 같다: **정답 클래스에 모델이 준 확률을 p라고 하면 손실 = −log(p)**. 정답을 높은 확률로 맞힐수록 손실이 작다.
  - 예) 정답이 3이고 모델이 3번에 0.9를 줬다면 −log(0.9) ≈ 0.105 (작음), 0.1을 줬다면 −log(0.1) ≈ 2.303 (큼)
- 그래서 같은 데이터에서 정답 형태만 맞추면 **결과(손실 값, 학습 결과)가 같다.**

### 왜 sparse가 따로 있나
1. **메모리/편의**: one-hot은 클래스가 많을수록 낭비가 크다. 클래스가 20,000개인 단어 분류라면 정답 하나가 길이 20,000짜리 벡터가 된다. 정수 하나로 주는 sparse가 훨씬 가볍다.
2. **변환 코드를 줄일 수 있다**: 숫자/문자를 이미 번호로 바꿔 둔 경우(이 노트북처럼) 바로 쓸 수 있다.
- 반대로 `categorical`은 정답이 이미 one-hot이거나, **label smoothing**처럼 정답 벡터를 `[0.9, 0.01, ...]`로 가공하고 싶을 때 쓴다 (sparse는 정수만 받음).

### 이 노트북에서
```python
y = np.flip(X, axis=1)                    # 정수 0~9
y = y.reshape((y.shape[0], y.shape[1], 1))  # (20000, 10, 1)  정수 번호 그대로
...
model.compile(..., loss='sparse_categorical_crossentropy', ...)
```
정답 `y`가 정수 번호이므로 sparse를 쓴다. one-hot으로 바꿔 쓰려면 `to_categorical(y, 10)`으로 y를 `(20000, 10, 10)`으로 만들고 loss를 `'categorical_crossentropy'`로 바꾸면 된다.

### 짝을 틀리면?
- 정답이 정수인데 `categorical_crossentropy`를 쓰거나, 정답이 one-hot인데 `sparse_...`를 쓰면 **shape 오류가 나거나 엉뚱한 학습**이 된다. loss 이름과 정답 형태를 항상 짝으로 기억해야 한다.

### 참고: 클래스가 2개일 때
- 클래스가 2개(이진 분류, 예: 고양이/강아지)면 출력 1개 + sigmoid + `binary_crossentropy`를 쓴다 (DogCat 노트북에서 사용한 방식).

cf. 강사님 설명
- sparse~는 내부적으로 one-hot encoding이 일어남 -> 정수 출력이라도 의미를 갖는 수로 사용되지 않음
- categorical~은 출력이 one-hot 형태..

---


## "10번 돈다"는 무슨 뜻일까? (`fit`, `predict`와 LSTM의 시간 축 반복)

> 강사님이 실제로 하신 말씀은 확인하지 못해서, 코드를 기준으로 가장 가능성이 높은 해석을 정리했다.
> 결론부터: **"10번"은 `epochs`나 `predict`를 10번 호출한다는 뜻이 아니라, LSTM이 숫자 10개를 한 칸씩 읽으며 안쪽에서 10번 반복한다는 뜻**일 가능성이 가장 크다.

### 1. 10의 정체: `InputNo = 10` = 시간 단계(time step) 수

```python
InputNo = 10
X = X.reshape((X.shape[0], X.shape[1], 1))      # (20000, 10, 1)
encoder_inputs = Input(shape=(InputNo, 1))       # (10, 1)
```

`(샘플 수, 10, 1)`의 의미는 이렇다.
- 첫 번째 `10` : **시간 단계가 10개** (숫자 10개가 순서대로 늘어서 있음)
- 마지막 `1` : 한 시점에 들어오는 값의 개수(특성 수) = 숫자 하나

LSTM은 이 중간 축(10)을 따라 **앞에서부터 한 칸씩 10번 읽는다.** 이것이 "10번 돈다"의 정체다.

> **주의: 이 노트북에는 "10"이 두 가지 의미로 나온다. 값이 같아서 헷갈리기 쉽다.**
>
> | 10의 의미 | 어디서 나오나 | `InputNo`와의 관계 |
> |---|---|---|
> | **시간 단계 수 (숫자 열의 길이)** | `InputNo = 10`, `Input(shape=(InputNo, 1))`, `reshape((N, 10, 1))` | **`InputNo` 때문에 생긴 10.** `InputNo`를 5로 바꾸면 5로 줄어든다. |
> | **클래스 수 (0~9, 열 가지 숫자)** | `randint(0, 10, ...)`, `Dense(10, activation='softmax')` | **`InputNo`와 무관.** `InputNo`를 5로 바꿔도 그대로 10이다. |
>
> 아래 2~3번에서 "시점 1 ~ 시점 10", "10칸", "10번"이라고 쓴 것은 모두 **시간 단계(InputNo)** 를 뜻한다.

### 2. 인코더 LSTM: 숫자 10개를 한 칸씩 10번 읽는다

```python
encoder = LSTM(128, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)
```

입력이 `[8, 4, 4, 4, 4, 2, 7, 8, 7, 1]`이라면:

```
시점 1 : 입력 8 + (초기 h, c = 0)   → 새 h1, c1
시점 2 : 입력 4 + (h1, c1)          → 새 h2, c2
시점 3 : 입력 4 + (h2, c2)          → 새 h3, c3
...
시점 10: 입력 1 + (h9, c9)          → 최종 h10, c10   ← 이게 state_h, state_c
```

- 같은 LSTM 가중치를 쓰면서, **직전 시점의 기억(h, c)을 다음 시점으로 넘기며** 10번 반복한다.
- 마지막(10번째) 시점의 h, c가 `state_h`, `state_c`로 나온다. 10개 숫자를 모두 읽은 뒤의 "요약"이다.
- 여기서 "10번"은 **시간 단계(InputNo)** 다. 위 예시 입력 `[8, 4, 4, 4, 4, 2, 7, 8, 7, 1]`의 숫자 개수가 10개라서 10번 읽는 것이다.
- `128`(LSTM 유닛 수)은 또 다른 값으로, 한 시점에 h, c가 담는 기억 벡터의 크기다. 반복 횟수와는 관계가 없다.

### 3. 디코더 LSTM: 출력도 10번 낸다

```python
decoder_lstm = LSTM(128, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
```

- 디코더도 입력이 `(10, 1)`이라 **10번 반복**한다. 매 시점마다 출력이 하나씩 나온다 (`return_sequences=True`라서 10개 모두 모아서 줌).
- 첫 시점의 시작 기억이 0이 아니라 **인코더의 최종 h, c** (`initial_state`).

```
시점 1 : 디코더 입력 0 + (인코더가 넘긴 h, c) → 출력1  → Dense → 예측 숫자 1번째
시점 2 : 디코더 입력 0 + (직전 h, c)          → 출력2  → Dense → 예측 숫자 2번째
...
시점 10: 디코더 입력 0 + (직전 h, c)          → 출력10 → Dense → 예측 숫자 10번째
```

그래서 최종 출력 shape이 `(샘플 수, 10, 10)`이다 (10개 위치 x 클래스 10개 확률).

> 이 shape의 두 10은 의미가 다르다.
> - **첫 번째 10** = 시간 단계 = 출력 위치 수 = `InputNo`. `InputNo=5`이면 5가 된다.
> - **두 번째 10** = 클래스 수(숫자 0~9) = `Dense(10)`. `InputNo`와 무관하게 그대로 10이다.
>
> 즉 `InputNo=5`이면 출력 shape은 `(샘플 수, 5, 10)`이다.

### 4. 그래서 `fit`과 `predict` 한 번은 안쪽에서 무슨 일을 하나?

```python
model.fit([X, decoder_input_data], y, epochs=50, ...)
predicted_sequence = model.predict([input_sequence, decoder_input])
```

**파이썬 코드에서는 한 줄이다.** 그런데 그 한 번의 호출 안에서:

| 호출 | 안쪽에서 일어나는 일 |
|---|---|
| `model.predict(...)` 1번 | 인코더가 10칸을 읽고 → 디코더가 10칸의 출력을 낸다. 즉 숫자 10개짜리 역순 결과가 **한 번에** 나온다. |
| `model.fit(...)` 샘플 1개 | 같은 계산(인코더 10칸 + 디코더 10칸)을 한 뒤, 오차를 **10칸을 거슬러 올라가며** 역전파한다 (BPTT, 시간에 따른 역전파). |

그래서 "특히 `fit`과 `predict` 코드에서 10번 돈다"는 설명은 **LSTM 층이 입력 길이(10)만큼 내부에서 순환한다**는 뜻으로 이해하면 맞다.

### 5. 헷갈리기 쉬운 다른 "반복"들과 구분

| 반복 | 횟수 | 무엇을 반복? |
|---|---|---|
| 시간 단계(time step) | **10** (`InputNo`) | LSTM이 숫자 10개를 한 칸씩 읽음 / 디코더가 출력 10개를 만듦 |
| 배치(batch) | 500 (= 학습 16000개 ÷ 배치 32) | 한 epoch 안에서 데이터를 32개씩 나눠 학습 (출력의 `500/500`이 이것) |
| epoch | 최대 50 (`EarlyStopping`으로 더 일찍 멈춤) | 전체 데이터를 처음부터 끝까지 한 번 학습 |

앞 노트북(1_1_10)에서는 `epochs=10`이었다. 그 "10"과 이번 "10(시간 단계)"은 **다른 10**이라서 헷갈릴 수 있다.

### 6. 이 코드가 "한 글자씩 10번 predict를 부르는" 방식이 아닌 이유

일반적인 Seq2Seq(번역 등)의 추론은 보통 이렇게 한다:

```
디코더에 <시작>을 넣어 첫 단어 예측 → 그 예측을 다음 입력으로 → 둘째 단어 예측 → ...
(predict를 단어 수만큼 반복 호출, 직전 예측을 다음 입력으로 되먹임)
```

그런데 이 노트북은:

```python
decoder_input = np.zeros((1, InputNo, 1))                          # 입력이 전부 0
predicted_sequence = model.predict([input_sequence, decoder_input])  # 한 번 호출로 10개 모두 예측
```

- 디코더 입력이 **전부 0**이라서 직전 예측을 다음 입력으로 넣을 필요가 없다. 매 시점 입력은 0으로 고정이고, 정보는 **오직 인코더가 넘긴 h, c와 디코더 자신의 기억**으로만 전달된다.
- 그래서 `predict`를 한 번만 불러도 10개가 한꺼번에 나온다. (진짜 "되먹임(autoregressive)" 방식의 Seq2Seq는 파이썬 `for` 문으로 10번 호출한다.)

> 만약 강사님이 "predict를 10번 반복해서 부른다"고 하셨다면, 그건 이 노트북 방식이 아니라 위의 일반적인 Seq2Seq 추론 방식을 설명하신 것일 수 있다. 이 코드에서는 `predict_reverse`를 부를 때마다 **모델 호출 1번에 안쪽 순환 10번**이다.

### 7. 한 줄 요약 (발표/답변용)

> "`InputNo=10`이라서 LSTM의 시간 단계가 10개입니다. 인코더는 숫자 10개를 한 칸씩 읽으며 기억(h, c)을 10번 갱신하고, 디코더는 그 기억에서 출발해 10칸의 출력을 만듭니다. `fit`이나 `predict`를 한 번 호출해도 LSTM 안쪽에서는 이 시간 축 반복이 10번 일어납니다."

### 확인용 질문 (선생님께 여쭤볼 거리)
- "10번 돈다"는 것이 **LSTM의 time step 10번**을 말씀하신 건가요, 아니면 `predict`를 10번 부르는 추론 방식을 말씀하신 건가요?

In [2]:
# 인코더 정의
encoder_inputs = Input(shape=(InputNo, 1))
encoder = LSTM(128, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)
encoder_states = [state_h, state_c]

# 디코더 정의
decoder_inputs = Input(shape=(InputNo, 1))
decoder_lstm = LSTM(128, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = Dense(10, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# 모델 정의
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

# loss='sparse_categorical_crossentropy' : 정답 y가 one-hot이 아니라 "정수 번호(0~9)"여서 sparse 버전을 쓴다.
#   (y는 위에서 (20000, 10, 1) 정수 배열로 만들었다. one-hot이었다면 categorical_crossentropy를 쓴다. 자세한 설명은 위 markdown 셀 참고)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 입력 데이터와 타겟 데이터를 동일하게 맞춤
decoder_input_data = np.zeros_like(X)

# EarlyStopping 콜백 정의
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 모델 학습
model.fit([X, decoder_input_data], y, epochs=50, validation_split=0.2, callbacks=[early_stopping])

Epoch 1/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 43s 55ms/step - accuracy: 0.2478 - loss: 1.9472 - val_accuracy: 0.4511 - val_loss: 1.4272
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 32s 63ms/step - accuracy: 0.6433 - loss: 1.0471 - val_accuracy: 0.8012 - val_loss: 0.7529
Epoch 3/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 40s 81ms/step - accuracy: 0.8536 - loss: 0.6009 - val_accuracy: 0.8829 - val_loss: 0.4874
Epoch 4/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 38s 75ms/step - accuracy: 0.9318 - loss: 0.3541 - val_accuracy: 0.9472 - val_loss: 0.2746
Epoch 5/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 32s 64ms/step - accuracy: 0.9645 - loss: 0.2045 - val_accuracy: 0.9798 - val_loss: 0.1433
Epoch 6/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 28s 56ms/step - accuracy: 0.9794 - loss: 0.1235 - val_accuracy: 0.9713 - val_loss: 0.1262
Epoch 7/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 48s 71ms/step - accuracy: 0.9815 - loss: 0.0932 - val_accuracy: 0.9848 - val_loss: 0.0775
Epoch 8/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 54s 108ms/step - accuracy: 0.9768 - loss: 0.0926 -

In [3]:

# 모델 검증 함수
def predict_reverse(input_sequence):
    input_sequence = np.array(input_sequence).reshape((1, InputNo, 1))
    decoder_input = np.zeros((1, InputNo, 1))
    predicted_sequence = model.predict([input_sequence, decoder_input])
    return np.argmax(predicted_sequence, axis=-1).reshape((InputNo,))
'''
# 입력 반복 루프
while True:
    user_input = input("입력 (쉼표로 구분된 5개의 숫자, 종료하려면 'quit' 입력): ")
    if user_input.lower() == "quit":
        break
    try:
        input_sequence = list(map(int, user_input.split(',')))
        if len(input_sequence) != 5:
            print("5개의 숫자를 입력해 주세요.")
            continue
        predicted_output = predict_reverse(input_sequence)
        print(f"입력: {input_sequence}")
        print(f"예측된 출력: {predicted_output}")
    except ValueError:
        print("유효한 숫자를 입력해 주세요.")
'''

'\n# 입력 반복 루프\nwhile True:\n    user_input = input("입력 (쉼표로 구분된 5개의 숫자, 종료하려면 \'quit\' 입력): ")\n    if user_input.lower() == "quit":\n        break\n    try:\n        input_sequence = list(map(int, user_input.split(\',\')))\n        if len(input_sequence) != 5:\n            print("5개의 숫자를 입력해 주세요.")\n            continue\n        predicted_output = predict_reverse(input_sequence)\n        print(f"입력: {input_sequence}")\n        print(f"예측된 출력: {predicted_output}")\n    except ValueError:\n        print("유효한 숫자를 입력해 주세요.")\n'

In [4]:
# 검증용 데이터 생성
test_X, test_y = generate_data(20)

# 모델 예측 및 정확도 계산
correct_predictions = 0
total_predictions = len(test_X)

for i in range(total_predictions):
    input_sequence = test_X[i]
    true_output = test_y[i].reshape(InputNo,)
    predicted_output = predict_reverse(input_sequence)
    
    if np.array_equal(predicted_output, true_output):
        correct_predictions += 1

    print(f"입력: {input_sequence.reshape(InputNo,)}")
    print(f"예측된 출력: {predicted_output}")
    print(f"실제 출력: {true_output}")
    print(f"정확 여부: {'맞음' if np.array_equal(predicted_output, true_output) else '틀림'}\n")

accuracy = correct_predictions / total_predictions
print(f"총 정확도: {accuracy * 100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
입력: [0 2 6 5 4 4 3 8 9 5]
예측된 출력: [5 9 8 3 4 4 5 6 2 0]
실제 출력: [5 9 8 3 4 4 5 6 2 0]
정확 여부: 맞음

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 546ms/step
입력: [0 0 1 7 3 2 5 3 8 1]
예측된 출력: [1 8 3 5 2 3 7 1 0 0]
실제 출력: [1 8 3 5 2 3 7 1 0 0]
정확 여부: 맞음

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 921ms/step
입력: [2 8 9 2 6 4 3 1 8 5]
예측된 출력: [5 8 1 3 4 6 2 9 8 2]
실제 출력: [5 8 1 3 4 6 2 9 8 2]
정확 여부: 맞음

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 588ms/step
입력: [5 4 9 2 7 8 0 7 4 0]
예측된 출력: [0 4 7 0 8 7 2 9 4 5]
실제 출력: [0 4 7 0 8 7 2 9 4 5]
정확 여부: 맞음

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step
입력: [0 1 3 2 1 7 7 7 8 8]
예측된 출력: [8 8 7 7 7 1 2 3 1 0]
실제 출력: [8 8 7 7 7 1 2 3 1 0]
정확 여부: 맞음

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 319ms/step
입력: [8 2 2 5 8 4 0 5 7 7]
예측된 출력: [7 7 5 0 4 8 5 2 2 8]
실제 출력: [7 7 5 0 4 8 5 2 2 8]
정확 여부: 맞음

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 236ms/step
입력: [7 2 4 2 5 3 0 3 6 9]
예측된 출력: [9 6 3 0 3 5 2 4 2 7]
실제 출력: [9 6 3 0 3 5 2 4 2 7]
정확 여부: 맞음

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 352ms/step
입력: [2 8 1 0 1 0 2 